<!-- dads-lab-header -->
<a href="https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M08/M08_Lab1_CrewAI_Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style="background: linear-gradient(135deg, #001a70 0%, #0055d4 100%); padding: 30px 36px; border-radius: 14px; margin-bottom: 20px;">
  <h1 style="color: #fff; margin: 0; font-size: 28px;">👥 M08 Lab 1 — CrewAI Basics</h1>
  <p style="color: rgba(255,255,255,0.85); margin: 8px 0 0; font-size: 15px;">DADS 5250: Generative AI in Practice &nbsp;|&nbsp; Prof. Dehghani</p>
  <p style="color: rgba(255,255,255,0.65); margin: 6px 0 0; font-size: 13px;">⭐ Difficulty: Intermediate &nbsp;|&nbsp; ⏱ Time: ~35 min</p>
</div>

> **📌 Note on models.** This lab references specific LLM versions through `DEFAULT_CHAT_MODEL` and `DEFAULT_MINI_MODEL` constants in the `dads5250` utility package. Models update quickly — feel free to swap to any newer OpenAI / Anthropic / Google model you have access to.


In [ ]:
# === Shared lab setup: install dads5250 + load API key + sticky pill ===
# Installs the shared utilities (pp, pretty_print, lab_pill, model constants,
# setup_openai, setup_gemini) once per Colab runtime. The same OPENAI_API_KEY
# / GEMINI_API_KEY Colab secrets are used across every DADS 5250 lab — set
# them once in the 🔑 sidebar and they're picked up automatically.
import os
import importlib.util
if importlib.util.find_spec("dads5250") is None:
    !pip install -q "git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils"

from dads5250 import (
    pp,
    pretty_print,
    lab_pill,
    setup_openai,
    setup_gemini,
    DEFAULT_CHAT_MODEL,   # newest reasoning model that supports temperature
    DEFAULT_MINI_MODEL,   # newest mini model that supports temperature
    DEFAULT_EMBED_MODEL,  # current embeddings default
    DEFAULT_GEMINI_MODEL, # tracks the latest stable flash
)

lab_pill('M08 Lab 1 — CrewAI Basics')            # sticky banner so you always see which lab you're in
client = setup_openai()        # loads OPENAI_API_KEY + verifies it works


<a href="https://colab.research.google.com/github/mdehghani86/AppliedGenAI/blob/main/M11_Lab1_CrewAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <title>CrewAI Lab</title>
  <style>
    body {
      font-family: 'Segoe UI', sans-serif;
      background-color: #f8fbff;
      color: #222;
      padding: 30px;
      line-height: 1.6;
    }

    h1, h2 {
      color: #1e4b8f;
    }

    a {
      color: #1e4b8f;
      text-decoration: none;
    }

    a:hover {
      text-decoration: underline;
    }

    .center {
      text-align: center;
    }

    .title-img {
      width: 600px;
      max-width: 100%;
      margin-bottom: 20px;
    }

    .table-container {
      overflow-x: auto;
      margin-top: 20px;
    }

    table {
      border-collapse: collapse;
      width: 100%;
      background-color: #fff;
      box-shadow: 0 2px 5px rgba(0,0,0,0.05);
    }

    th, td {
      padding: 12px 16px;
      text-align: left;
      border-bottom: 1px solid #e0e0e0;
    }

    th {
      background-color: #e6f0fb;
      color: #1e4b8f;
    }

    ul {
      margin-top: 10px;
    }
  </style>
</head>
<body>

  <div class="center">
    <img src="https://www.dropbox.com/scl/fi/72c3k12bjnhxdebypttum/crewAI_title.png?dl=1" alt="crewAI Title" class="title-img">
  </div>

  <h1>Welcome to the CrewAI Lab</h1>

  <p>
    This lab introduces <strong>CrewAI</strong>, a powerful platform for orchestrating collaborative AI agents, and explores its integration with LangChain and n8n.
    Visit the <a href="https://www.crewai.com/" target="_blank">CrewAI website</a> to learn more about how CrewAI empowers teams to build complex, automated workflows and advanced_


## **CryptoTrio: A Collaborative Model**

In this **CryptoTrio** model, we bring together three specialized agents to handle different aspects of Bitcoin investment decisions:

1. **Market Analyst** – Gathers and interprets market data, recent trends, and news sentiment.
2. **Risk Evaluator** – Assesses volatility, identifies potential threats, and evaluates overall market risks.
3. **Decision Maker** – Synthesizes the analyst’s findings and the risk assessment to provide a final recommendation (Buy, Sell, or Hold).

<p align="center">
  <img src="https://www.dropbox.com/scl/fi/4lz7hgl6me2ymb51dfd8r/Flow.png?rlkey=yqtw7izjpxxqg2047ip58r9ly&dl=1" alt="CryptoTrio Flow" style="width:200px;">
</p>

In [ ]:
%%capture
# 🚀 Installing CrewAI, LangChain, OpenAI, and additional tools/libraries quietly...
!pip install -q crewai langchain langchain-openai openai crewai-tools ipython
!pip install 'crewai[tools]'


In [ ]:
# ### 🔑 API Keys and Model Configuration Cell
import os
from google.colab import userdata

# Load OpenAI API key from Colab userdata secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

# Set default OpenAI model for CrewAI agents
os.environ["OPENAI_MODEL_NAME"] = DEFAULT_MINI_MODEL

# Confirmation message
print("✅ OpenAI key and model are successfully configured.")


## 🤖 Defining Agents in CrewAI

In CrewAI, each **agent** represents an autonomous expert responsible for a specific task.  
You define agents using three main components:

- **Role 🎯:** Clearly describes the agent's professional role or expertise.
- **Goal 🎯:** Defines the specific objective the agent should achieve.
- **Backstory 📖:** Provides a short narrative about the agent’s expertise or experience, giving context to their behavior.

In [ ]:
from crewai import Agent, LLM

# 🤖 Define the language model instance (LLM)
llm_model = LLM(model=DEFAULT_MINI_MODEL)

# 1️⃣ Define the Market Analyst agent
market_analyst = Agent(
    role="Market Analyst",
    goal="Analyze current Bitcoin market data, trends, and news sentiment.",
    backstory="An expert cryptocurrency analyst skilled in interpreting market movements.",
    llm=llm_model,
    verbose=True  # Enables detailed logging
)

# 2️⃣ Define the Risk Evaluator agent
risk_evaluator = Agent(
    role="Risk Evaluator",
    goal="Evaluate risks, volatility, and potential threats in Bitcoin investments.",
    backstory="Specialist in financial risk management with extensive experience in crypto markets.",
    llm=llm_model,
    verbose=True  # Enables detailed logging
)

# 3️⃣ Define the Decision Maker agent
# This agent provides clear investment recommendations (Buy, Sell, or Hold) and includes detailed analytics.
decision_maker = Agent(
    role="Investment Decision Maker",
    goal="Provide clear investment recommendations: Buy, Sell, or Hold Bitcoin, with detailed analytics supporting the decision.",
    backstory="Experienced investment strategist capable of balancing risks and market opportunities by integrating comprehensive market analytics into the final decision.",
    llm=llm_model,
    verbose=True  # Enables detailed logging
)


In [ ]:
from crewai import Task

# 1️⃣ Task for the Market Analyst agent
market_analysis_task = Task(
    description=(
        "Analyze current Bitcoin market prices, trends, and recent news sentiment. "
        "Provide a concise summary of current market conditions."
    ),
    expected_output=(
        "A clear summary of Bitcoin prices, recent market trends, and sentiment analysis from news sources."
    ),
    agent=market_analyst
)

# 2️⃣ Task for the Risk Evaluator agent
risk_assessment_task = Task(
    description=(
        "Evaluate the current market risks and potential volatility associated with investing in Bitcoin today. "
        "Clearly identify potential threats."
    ),
    expected_output=(
        "A detailed risk assessment including volatility indicators and clearly listed potential threats for investing in Bitcoin."
    ),
    agent=risk_evaluator
)

# 3️⃣ Task for the Decision Maker agent
investment_decision_task = Task(
    description=(
        "Based on the market analysis and risk evaluation, recommend an investment action for Bitcoin: Buy, Sell, or Hold. "
        "Include detailed analytics and reasoning behind your decision."
    ),
    expected_output=(
        "A clear investment recommendation (Buy, Sell, or Hold) with comprehensive analytics supporting the decision."
    ),
    agent=decision_maker
)


In [ ]:
# ### Crew Creation and Execution Cell
# This cell creates a Crew with all defined agents and tasks, executes the crew,
# and then displays the final recommendation in a nicely formatted Markdown cell.

from crewai import Crew
from IPython.display import display, Markdown

# 🔹 Create the crew with the existing agents and tasks
bitcoin_investment_crew = Crew(
    agents=[market_analyst, risk_evaluator, decision_maker],
    tasks=[market_analysis_task, risk_assessment_task, investment_decision_task],
    verbose=True
)

# 🔸 Kick off the crew and capture the final recommendation
final_recommendation = bitcoin_investment_crew.kickoff()

# 🔷 Display the final recommendation in a formatted Markdown cell for clarity
display(Markdown("### Final Recommendation"))
display(Markdown(final_recommendation.raw)) # Access the 'raw' attribute of CrewOutput


# 👋 Hands-On Exercise

**Task:**  
Change the language model for the Market Analyst agent to a stronger one (e.g. set `llm_model = LLM(model=DEFAULT_CHAT_MODEL)` to use the default reasoning model defined in `utils.py`, or any newer OpenAI model you have access to), run the crew, and note any changes in the final recommendation.

**Your Observations:**  
> *[Add your observations here]*


# 💡 Introduction to CrewAI Tools

CrewAI provides a suite of **tools** that extend your agents' abilities by allowing them to interact with external data sources. These tools enable functions like web scraping, real-time data retrieval, and targeted searches—helping agents deliver more accurate and insightful outputs.

- **Extend Functionality:** Tools help agents do more than text generation by connecting them with live data.
- **Seamless Integration:** Easily attach these tools to your agents to enhance their decision-making process.
- **Examples:** Explore tools like *SerperDevTool* for general web searches or *ScrapeWebsiteTool* for content extraction.

For more details, refer to the [CrewAI Tools Documentation](https://docs.crewai.com/concepts/tools).

# 🚀 Agent Update
Let's update the Risk Analysis agent by integrating a Bitcoin API tool (e.g., CoinGecko) to fetch real-time Bitcoin prices and market data. This live data will enable the agent to perform more accurate, up-to-date risk assessments reflecting current market conditions.


In [ ]:
# ### 🪙 Bitcoin Data Tool using CoinGecko API
# This tool fetches Bitcoin market data from CoinGecko.
# It accepts:
# - 'days': (optional) Number of days of historical data to retrieve (default is 10).
# - 'detailed': (optional) If True, returns detailed hourly/daily data (first 10 data points with timestamps).
# Otherwise, it returns a summary with the start and end prices.

import requests
from crewai.tools import BaseTool
from datetime import datetime

class BitcoinDataTool(BaseTool):
    name: str = "Get Bitcoin Data 🪙"
    description: str = (
        "Fetches Bitcoin market data from CoinGecko. "
        "Optionally accepts a 'days' parameter (default 10) for historical data and a 'detailed' flag for detailed output."
    )

    def _run(self, *args, **kwargs) -> str:
        try:
            # Retrieve parameters; default to 10 days and summary output.
            days = kwargs.get("days", "10")
            detailed = kwargs.get("detailed", False)
            # Construct the URL for the CoinGecko market_chart endpoint.
            url = f"https://api.coingecko.com/api/v3/coins/bitcoin/market_chart?vs_currency=usd&days={days}"
            response = requests.get(url)
            data = response.json()
            prices = data.get("prices", [])
            if not prices:
                return "No price data available."
            if detailed:
                # Detailed output: show the first 10 data points with formatted timestamps.
                detailed_output = "Detailed Bitcoin Price Data:\n"
                for idx, point in enumerate(prices):
                    timestamp, price = point
                    dt = datetime.fromtimestamp(timestamp / 1000.0)
                    detailed_output += f"{dt.strftime('%Y-%m-%d %H:%M:%S')} - ${price:.2f} USD\n"
                    if idx == 9:
                        detailed_output += f"... and {len(prices) - 10} more data points."
                        break
                return detailed_output
            else:
                # Summary output: display only the first and last price.
                start_price = prices[0][1]
                end_price = prices[-1][1]
                return (
                    f"Bitcoin Price Data for the last {days} days:\n"
                    f"Start Price: ${start_price:.2f} USD\n"
                    f"End Price: ${end_price:.2f} USD"
                )
        except Exception as e:
            return f"Error fetching Bitcoin data: {e}"

# ----- Test the BitcoinDataTool function -----
# Test with detailed output (change 'detailed' to False for summary output)
test_output = BitcoinDataTool()._run(days="5", detailed=True)
pretty_print(str(test_output), title="🤖 Crew Output")


In [ ]:
import requests
import plotly.express as px
import pandas as pd
from datetime import datetime

# ----- Retrieve Bitcoin Data for Plotting -----
# We'll fetch the historical Bitcoin price data for the last 5 days.
days = "5"
url = f"https://api.coingecko.com/api/v3/coins/bitcoin/market_chart?vs_currency=usd&days={days}"
response = requests.get(url)
data = response.json()
prices = data.get("prices", [])

# ----- Process the Data into a DataFrame -----
# Convert each data point into a dictionary with a datetime and price.
data_list = [
    {"Date": datetime.fromtimestamp(ts / 1000.0), "Price": price}
    for ts, price in prices
]
df = pd.DataFrame(data_list)

# ----- Plot the Data using Plotly -----
# Create a line chart with Date on the x-axis and Price on the y-axis.
fig = px.line(df, x="Date", y="Price", title="Bitcoin Price Over the Last 5 Days")
fig.show()


In [ ]:
# ### 🪙 Simple Bitcoin Data Tool
# This tool fetches historical Bitcoin price data from CoinGecko for a specified number of days (default is 5)
# and returns two lists: one with human-readable timestamps and one with the corresponding prices.

import requests
from crewai.tools import BaseTool
from datetime import datetime

class BitcoinSimpleDataTool(BaseTool):
    name: str = "Get Bitcoin Simple Data 🪙"
    description: str = (
        "Fetches Bitcoin historical price data from CoinGecko for a given number of days "
        "(default is 5) and returns two lists: timestamps and prices."
    )

    def _run(self, *args, **kwargs) -> str:
        try:
            # Retrieve 'days' parameter; default to 5 days if not provided.
            days = kwargs.get("days", "5")
            url = f"https://api.coingecko.com/api/v3/coins/bitcoin/market_chart?vs_currency=usd&days={days}"
            response = requests.get(url)
            data = response.json()
            prices = data.get("prices", [])
            if not prices:
                return "No data available."

            # Initialize lists for timestamps and prices.
            timestamps = []
            price_values = []

            # Process each data point to extract the timestamp and price.
            for ts, price in prices:
                dt = datetime.fromtimestamp(ts / 1000.0).strftime("%Y-%m-%d %H:%M:%S")
                timestamps.append(dt)
                price_values.append(price)

            # Return the lists as a formatted string.
            return f"Timestamps: {timestamps}\nPrices: {price_values}"
        except Exception as e:
            return f"Error fetching Bitcoin data: {e}"

# ----- Test the BitcoinSimpleDataTool function -----
# Test the tool for a 5-day period.
test_output = BitcoinSimpleDataTool()._run(days="5")
pretty_print(str(test_output), title="🤖 Crew Output")


In [ ]:
# --- Update Risk Analysis Agent with the BitcoinSimpleDataTool ---
# Here we update the Risk Analysis agent (formerly Risk Evaluator) to use the BitcoinSimpleDataTool.
# This tool fetches live Bitcoin data as two lists (timestamps and prices) for a given number of days.
# The task is updated to instruct the agent to use the tool with a user-specified 'days' input.

# Update the Risk Analysis agent with the tool.
risk_evaluator = Agent(
    role="Risk Analysis Agent",
    goal="Evaluate current market risks and potential volatility in Bitcoin investments using live data.",
    backstory=(
        "A seasoned risk analysis expert who leverages live Bitcoin data to assess market volatility "
        "and potential threats. Now enhanced with a tool that retrieves real-time price data over a specified period."
    ),
    tools=[BitcoinSimpleDataTool()],  # Ensure BitcoinSimpleDataTool is defined in a previous cell.
    llm=llm_model,
    verbose=True
)

# Update the task for the Risk Analysis agent.
risk_assessment_task = Task(
    description=(
        "Evaluate the current market risks and potential volatility in Bitcoin investments. "
        "Use the Bitcoin data tool to fetch live price data for a specified number of days (input 'days'). "
        "Provide a detailed risk assessment based on this live data."
    ),
    expected_output="A detailed risk assessment including analysis of live Bitcoin price trends over the specified period.",
    agent=risk_evaluator
)

# --- Rebuild the Crew and Kick Off ---
# Reinitialize the crew with the updated Risk Analysis agent and its task,
# along with the unchanged Market Analyst and Decision Maker agents and tasks.
from crewai import Crew

bitcoin_investment_crew = Crew(
    agents=[market_analyst, risk_evaluator, decision_maker],
    tasks=[market_analysis_task, risk_assessment_task, investment_decision_task],
    verbose=True
)

# Kick off the crew.
final_recommendation = bitcoin_investment_crew.kickoff()
# 🔷 Display the final recommendation in a formatted Markdown cell for clarity
display(Markdown("### Final Recommendation"))
display(Markdown(final_recommendation.raw)) # Access the 'raw' attribute of CrewOutput


# 🤝 Agent Collaboration in CrewAI

CrewAI agents collaborate by sharing information, delegating tasks, and coordinating their efforts. This is achieved through several key features:

- **Allow Delegation:**  
  Agents can pass context or delegate subtasks to one another by setting `allow_delegation: true`.  
  *Example:* A strategist agent can delegate a data gathering subtask to a research agent.  
  ```python
  research_agent = Agent(
      role="Research Specialist",
      goal="Gather detailed market data from authoritative sources.",
      backstory="A meticulous researcher with a background in data analysis.",
      allow_delegation=True,  # Enables delegation
      llm=llm_model,
      verbose=True
  )


# 🤝 Collaboration Overview: Strategist & Risk Analysis Agents  

In our CrewAI workflow, we introduce a Bitcoin Trading Strategist agent whose role is to generate innovative trading strategies (e.g., "buy if price drops 5%, sell if it goes up 5%") using real-time online data. This agent works collaboratively with the Risk Analysis Agent as follows:  

- **Delegation:**  
  The Risk Analysis Agent (with `allow_delegation=True`) delegates the task of formulating additional strategies to the Strategist. This ensures that, beyond its default strategy, the Risk Analysis Agent can incorporate diverse perspectives.  

- **Orchestration:**  
  The Crew orchestrates the execution so that the Strategist produces 2 innovative strategies. The Risk Analysis Agent then integrates these strategies with its own default approach to provide a comprehensive risk evaluation.  

This cooperative setup enhances decision-making by combining specialized strategy generation with robust risk assessment.  

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

from crewai import Agent, LLM, Task, Crew
from crewai_tools import WebsiteSearchTool  # Tool for online data retrieval
from IPython.display import display, Markdown

# --- Use OpenAI directly for advanced LLM (e.g., GPT-4) ---
advanced_llm = LLM(model=DEFAULT_CHAT_MODEL)

# --- Create the Website Search Tool instance ---
# This tool will fetch online strategy info from the given URL.
website_rag_tool = WebsiteSearchTool(search_url="https://www.investopedia.com/investing-in-crypto-6502543")

# --- Define the Bitcoin Trading Strategist Agent ---
# This agent proposes additional innovative trading strategies.
# Delegation is NOT enabled for this agent.
strategist_agent = Agent(
    role="Bitcoin Trading Strategist",
    goal="Propose 2 additional innovative Bitcoin trading strategies based on external research.",
    backstory=(
        "A seasoned market strategist with deep expertise in Bitcoin trading and risk management. "
        "This agent leverages real-time online data via a Website Search Tool to generate innovative strategies. "
        "It does not delegate tasks further."
    ),
    tools=[website_rag_tool],
    llm=advanced_llm,
    verbose=True,
    allow_delegation=False
)

# --- Update the Risk Analysis Agent ---
# This agent is allowed to delegate. It uses its own default strategy (e.g., buy if price drops 5%, sell if price rises 5%)
# and delegates additional strategy generation to the Bitcoin Trading Strategist.
risk_evaluator = Agent(
    role="Risk Analysis Agent",
    goal="Evaluate market risks and decide on trading actions using a default strategy and additional delegated strategies.",
    backstory=(
        "An expert in risk management who relies on live data to assess market conditions. "
        "This agent uses a default strategy—buy if the price drops 5% or sell if the price rises 5%—and delegates "
        "the task of generating extra innovative strategies to the Bitcoin Trading Strategist."
    ),
    llm=advanced_llm,
    verbose=True,
    allow_delegation=True  # Enables delegation so it can consult the strategist.
)

# --- Assume the Market Analyst and Decision Maker agents (and their tasks) are defined elsewhere ---
# For this example, we use the pre-defined 'market_analyst' and 'decision_maker'.

# --- Update task for the Strategist Agent ---
strategist_task = Task(
    description=(
        "Using the Website Search Tool, provide 2 additional innovative Bitcoin trading strategies "
        "(apart from the default: 'buy if price drops 5%, sell if price rises 5%'). "
        "Include brief explanations for each strategy."
    ),
    expected_output="A list of 2 additional trading strategies with brief explanations.",
    agent=strategist_agent
)

# --- Update task for the Risk Analysis Agent ---
risk_assessment_task = Task(
    description=(
        "Evaluate the current market risks using your default strategy (buy if price drops 5%, sell if price rises 5%). "
        "Then delegate to the Bitcoin Trading Strategist to obtain 2 extra strategies and integrate them into your analysis. "
        "Provide a detailed risk assessment based on all these strategies."
    ),
    expected_output="A detailed risk evaluation incorporating the default strategy and additional strategies from the strategist.",
    agent=risk_evaluator
)

# --- Combine all agents and tasks ---
# (Assuming market_analyst, risk_evaluator, decision_maker, market_analysis_task, and investment_decision_task are defined)
all_agents = [market_analyst, risk_evaluator, decision_maker, strategist_agent]
all_tasks = [market_analysis_task, risk_assessment_task, investment_decision_task, strategist_task]

# --- Create the Crew and Kick It Off ---
bitcoin_investment_crew = Crew(
    agents=all_agents,
    tasks=all_tasks,
    verbose=True
)

final_recommendation = bitcoin_investment_crew.kickoff()

# --- Display the Final Output ---
display(Markdown("### Final Recommendation and Collaborative Output"))
display(Markdown(final_recommendation.raw))
